# Assignment 5 — SVJD Simulation and Distributional Risk


**Student:** YOUR NAME HERE  
**Course:** Machine Learning in Finance and Macroeconomics  
**Submission:** Submit the completed `.ipynb` notebook.

### Instructions

1. Complete every **Core Exercise**.
2. Keep the question text in the notebook.
3. Put code in the designated code cells and short explanations in the designated Markdown cells.
4. Before submitting, use **Restart Kernel and Run All Cells**.
5. Make sure all requested tables and figures appear in the saved notebook.
6. The notebook should run from top to bottom without relying on variables from another notebook.
7. Unless an exercise says otherwise, use the S&P 500 data downloaded in the Setup section.

**Suggested filename:** `Lastname_Firstname_ClassXX_Assignment.ipynb`


## Setup and central idea

The key distinction for this assignment is:

> **An SVJD simulation is a draw from an estimated stochastic process. It is not a date-by-date forecast of the realized S&P 500 path.**

- We will therefore evaluate anohter index using distributional moments, quantiles, and tail-risk measures.
- Please replace the ^GSPC with another ticker symbol from Yahoo finance.


In [12]:
   # If needed:
# %pip install yfinance arch scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import course_tools_general as ctg


# Download data for any ticker directly into your target variable
spx = ctg.load_prices(ticker="^GSPC", start="1990-01-01")
spx.head()

print(f"S&P 500 sample: {spx.index.min().date()} to {spx.index.max().date()}")

# These range measures were introduced in Class 2.
# Reuse them here without requiring the Class 2 kernel to remain open.
gk20 = ctg.garman_klass_rolling(spx, window=20).to_numpy()
ok20 = ctg.ok_rolling(spx, window=20).to_numpy()




# GJR-GARCH-in-Mean was taught in Class 3.  Re-estimate it here quietly
# so this notebook can be run independently after a kernel restart.
gjr_daily_vol, garch_m_lev_results = ctg.fit_gjr_garch_in_mean_volatility(spx, nlags=5)





S&P 500 sample: 1990-01-02 to 2026-08-28


## Provided functions

The empirical-moment function and the corrected SVJD simulator are supplied. Notice that if \(N_t>1\), the jump component is the **sum of \(N_t\) independent jump sizes**.


In [6]:
from scipy.stats import skew, kurtosis
def empirical_moments(r):
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]

    return np.array([
        np.mean(r),
        np.var(r),
        skew(r),
        kurtosis(r, fisher=False),
        np.corrcoef(r[1:], r[:-1])[0, 1],
        np.corrcoef(r[1:]**2, r[:-1]**2)[0, 1],
        np.mean(np.abs(r)),
        np.percentile(r, 1),
        np.percentile(r, 99)
    ])


def simulate_svjd(params, T, dt=1/252, seed=None):
    rng = np.random.default_rng(seed)

    mu, kappa, theta, sigma_v, rho, lam, mu_j, sigma_j = params

    v = np.zeros(T)
    r = np.zeros(T)
    v[0] = theta

    for t in range(1, T):
        z1 = rng.normal()
        z2 = rho * z1 + np.sqrt(1 - rho**2) * rng.normal()

        N = rng.poisson(lam * dt)

        if N > 0:
            # Sum of N independent Normal(mu_j, sigma_j^2) jumps
            jump = rng.normal(N * mu_j, np.sqrt(N) * sigma_j)
        else:
            jump = 0.0

        v[t] = (
            v[t-1]
            + kappa * (theta - v[t-1]) * dt
            + sigma_v * np.sqrt(max(v[t-1], 0.0)) * np.sqrt(dt) * z2
        )
        v[t] = max(v[t], 1e-8)

        r[t] = (
            (mu - 0.5 * v[t-1]) * dt
            + np.sqrt(v[t-1]) * np.sqrt(dt) * z1
            + jump
        )

    return r[1:], v[1:]


moment_names = [
    "Mean", "Variance", "Skewness", "Kurtosis",
    "Return autocorr", "Squared-return autocorr",
    "Mean abs return", "1% quantile", "99% quantile"
]


## Core Exercise 1 — Historical distributional moments

1. Apply `empirical_moments` to the historical S&P 500 returns and to another indwx.
2. Display the nine moments in a labeled table.
3. Identify the moments that indicate non-normality and volatility dependence.

**Question:** Why are squared-return autocorrelation and tail quantiles useful moments for an SVJD model?


In [11]:
r = spx["Return"]
result = empirical_moments(r)
moment_names = [
    "Mean",
    "Variance",
    "Skewness",
    "Kurtosis",
    "Return autocorrelation",
    "Squared return autocorrelation",
    "Mean absolute return",
    "1% percentile",
    "99% percentile"
]

results_table = pd.DataFrame({
    "Moment": moment_names,
    "Value": result
})

display(results_table)


,Moment,Value
0,Mean,0.000332
1,Variance,0.000129
2,Skewness,-0.364107
3,Kurtosis,13.872634
4,Return autocorrelation,-0.078938
5,Squared return autocorrelation,0.310524
6,Mean absolute return,0.007572
7,1% percentile,-0.031539
8,99% percentile,0.029800


### Your interpretation

*Replace this text with a short answer.*


## Core Exercise 2 — Simulate an SVJD distribution

Use the following parameter vector as a **demonstration parameterization**:

`[0.08, 3.0, 0.04, 0.40, -0.50, 0.50, -0.02, 0.05]`

1. Simulate at least 100,000 daily observations after a burn-in of 5,000 observations.
2. Compute the same nine moments.
3. Put historical and simulated moments side by side.

**Question:** Which moments are reproduced reasonably well and which are not? Remember: the goal is distributional comparison, not matching historical dates.


In [ ]:
params_demo = np.array([
    0.08,   # mu
    3.0,    # kappa
    0.04,   # theta
    0.40,   # sigma_v
    -0.50,  # rho
    0.50,   # annual jump intensity
    -0.02,  # mean jump size
    0.05    # jump-size standard deviation
])

# YOUR CODE HERE


### Your interpretation

*Replace this text with a short answer.*


## Core Exercise 3 — Distribution and quantile diagnostics

Using the long simulated sample from Exercise 2:

1. Plot a KDE/density estimate for historical and simulated returns on the same axes.
2. Create a quantile-to-quantile comparison using probabilities from 0.01 to 0.99.
3. Include a 45-degree line on the quantile plot.

**Question:** Why is this comparison more meaningful than plotting the simulated sequence against historical returns by calendar date?


In [ ]:
# YOUR CODE HERE


### Your interpretation

*Replace this text with a short answer.*


## Core Exercise 4 — Distributional tail risk

From the simulated SVJD distribution compute:

- 5% lower-tail return quantile,
- 1% lower-tail return quantile,
- Expected Shortfall at 5%,
- Expected Shortfall at 1%,
- probability of a daily return below \(-5\%\).

Compute the same quantities from the historical sample where meaningful and put them in a comparison table.

**Question:** What information does Expected Shortfall provide that VaR does not?


In [ ]:
# YOUR CODE HERE


### Your interpretation

*Replace this text with a short answer.*


## Core Exercise 5 — Interpretation: simulation versus forecasting

In **4–6 sentences**, explain the difference between:

1. a conditional GARCH volatility forecast, and
2. a long SVJD Monte Carlo simulation used to characterize the fitted distribution.

Your answer must explicitly discuss **calendar-time alignment** and **distributional inference**.


### Your answer

*Replace this text with your 4–6 sentence answer.*


## Optional Extension — Simulated GMM

Using the moment vector above, construct a simple simulated-GMM objective and use `scipy.optimize.minimize` to improve the demonstration parameter vector.

To keep runtime manageable, you may use only 2–3 simulation replications and a simulation length of roughly 5 times the historical sample during optimization. After estimation, evaluate the fitted parameters with a much longer independent simulation.

**Do not judge the fitted model by date-by-date path matching.**


In [ ]:
# OPTIONAL CODE HERE


## Submission checklist

- [ ] Historical and simulated moments compared
- [ ] Density and quantile diagnostics included
- [ ] VaR and Expected Shortfall computed
- [ ] Simulation-versus-forecasting distinction explained correctly
- [ ] No simulated path is treated as a forecast of historical dates
- [ ] Restarted kernel and ran all cells successfully
